In [ ]:
from dataclasses import replace

import torch
import numpy as np
import matplotlib.pyplot as plt

from pathtracer import Renderer, lattice_preview

from spectral import lattice_geometry, row_twiddle, fft, ifft, bin_freq, alias_radius_sq
from utils import device_mapper, host

In [ ]:
DEVICE = device_mapper()

In [ ]:
scene = Renderer("scenes/cornell.json")
rendercam = replace(scene.default_camera)
halves = torch.from_numpy(np.stack([scene.render(camera=rendercam, aovs=("luminance",), width=64, height=32, samples=scene.default_samples // 2, seed=seed, show_sky=True)["luminance"][..., 0] for seed in (1, 2)])).to(DEVICE)

In [ ]:
plt.imshow(lattice_preview(host(halves.mean(0)[..., None]), rendercam, scale=8))
plt.xticks([]), plt.yticks([])
plt.tight_layout()

In [ ]:
# luminance > log > noise-aware whitening > oriented filter bank (Gabor or learned) → sparsifying step
dark = ((halves[0] - halves[1]) ** 2 / 4).mean().sqrt()
log_x = torch.log(halves + dark)

In [ ]:
row_pitch, row_shift = lattice_geometry(rendercam)
twiddle = row_twiddle(halves.shape[-2:], row_shift, DEVICE)
fy, fx = bin_freq(halves.shape[-2:], row_pitch, DEVICE)
radius_sq, true_fy, true_fx = alias_radius_sq(fy, fx, row_pitch, row_shift)
radius = radius_sq.sqrt()
ac = radius > 0
lowest = 1 / min(halves.shape[-1], halves.shape[-2] * row_pitch)
ring = (radius / lowest).long()
count = torch.zeros(int(ring.max()) + 1, device=DEVICE).index_add_(0, ring.flatten(), ac.flatten().float())
ring_mean = lambda v: torch.zeros_like(count).index_add_(0, ring.flatten(), torch.where(ac, v, 0).flatten()) / count

In [ ]:
first, other = fft(log_x, twiddle)
signal = ring_mean((first * other.conj()).real).clamp(min=0)
noise = ring_mean((first - other).abs() ** 2 / 4)
gain = signal.sqrt() / (signal + noise)
transfer = torch.where(ac, gain[ring], 0)
whitened = (first + other) / 2 * transfer

In [ ]:
# G2 (cos²) and its Riesz quadrature H2 (cos³) share one radial profile, so hypot is phase-invariant edge energy at every theta
n = 36
theta = torch.pi * torch.arange(n, device=DEVICE)[:, None, None] / n
peak = 1 / 4  # cycles per pixel: Sobel's peak, lambda = 4 px
radial = radius ** 2 * torch.exp(-(radius / peak) ** 2)
cos = torch.cos(torch.atan2(true_fy, true_fx) - theta)
g2 = ifft(whitened * radial * cos ** 2, twiddle).real
h2 = ifft(whitened * radial * cos ** 3, twiddle).imag
edges = torch.hypot(g2, h2).amax(0)

plt.figure(figsize=(10, 5))
plt.imshow(lattice_preview(host(edges[..., None]), rendercam)[..., 0], cmap="gray")
plt.title("G2/H2")
plt.xticks([]), plt.yticks([])
plt.tight_layout()